<a href="https://colab.research.google.com/github/olaidczak/dl-project-1/blob/twardowski_branch/dl_project_version6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
!pip uninstall -y sympy
!pip install sympy==1.13.3
from torch import nn
from torch.utils.data import Subset, DataLoader
from torchvision import datasets, transforms, models
import torch.optim as optim
import requests
import random
import kagglehub
import datetime
import os
import json
import numpy as np
from datetime import datetime
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from collections import defaultdict
from timeit import default_timer as timer
from google.colab import runtime


Found existing installation: sympy 1.13.3
Uninstalling sympy-1.13.3:
  Successfully uninstalled sympy-1.13.3
  Using cached sympy-1.13.3-py3-none-any.whl.metadata (12 kB)
Using cached sympy-1.13.3-py3-none-any.whl (6.2 MB)


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
seed = 134
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
# Deterministyczne wyniki GPU (fix #3)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device


'cuda'

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
image_path = Path(kagglehub.dataset_download("mengcius/cinic10"))
train_dir = image_path / "train"
valid_dir = image_path / "valid"
test_dir = image_path / "test"

100%|██████████| 754M/754M [00:05<00:00, 137MB/s]

Extracting files...


In [5]:
# Statystyki CINIC-10 (inne niz ImageNet i CIFAR-10!)
CINIC_MEAN = [0.47889522, 0.47227842, 0.43047404]
CINIC_STD  = [0.24205776, 0.23828046, 0.25874835]

# Transform treningowy: RandomCrop + HorizontalFlip + Normalize (fix #6)
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=CINIC_MEAN, std=CINIC_STD)
])

# Transform walidacyjny/testowy: BRAK augmentacji, tylko normalize
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=CINIC_MEAN, std=CINIC_STD)
])

train_data = datasets.ImageFolder(root=train_dir, transform=train_transform)

# fix #1: valid i test to osobne obiekty z osobnych folderow
valid_data = datasets.ImageFolder(root=valid_dir, transform=eval_transform)
test_data  = datasets.ImageFolder(root=test_dir,  transform=eval_transform)


In [6]:
def build_resnet18(num_classes=10, weights="IMAGENET1K_V1", dropout=0):
  model = models.resnet18(weights=weights)
  # zmiany ktore pozwalją lepiej dzialac na obrazkach 32x32, bo defaultowo sa 224x224
  model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
  model.maxpool = nn.Identity()

  # Add dropout
  model.fc = nn.Sequential(
    nn.Dropout(dropout),
    nn.Linear(model.fc.in_features, num_classes)
)
  return model

In [7]:
def build_densenet121(num_classes=10, weights="IMAGENET1K_V1", dropout=0):
    model = models.densenet121(weights=weights)

    # Adjust for 32x32 images
    model.features.conv0 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.features.pool0 = nn.Identity()

    # Add dropout
    num_ftrs = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Dropout(dropout),
        nn.Linear(num_ftrs, num_classes)
    )

    return model

In [8]:
def train_step(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer):
    model.train()
    train_loss, train_acc = 0, 0
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        y_pred = model(X)
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        train_acc += (y_pred_class == y).sum().item()/len(y_pred)

    train_loss = train_loss / len(dataloader)
    train_acc = train_acc / len(dataloader)
    return train_loss, train_acc

def test_step(model: torch.nn.Module,
              dataloader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module):
    model.eval()
    test_loss, test_acc = 0, 0

    with torch.inference_mode():
        for batch, (X, y) in enumerate(dataloader):
            X, y = X.to(device), y.to(device)
            test_pred_logits = model(X)
            loss = loss_fn(test_pred_logits, y)
            test_loss += loss.item()
            test_pred_labels = test_pred_logits.argmax(dim=1)
            test_acc += ((test_pred_labels == y).sum().item()/len(test_pred_labels))

    test_loss = test_loss / len(dataloader)
    test_acc = test_acc / len(dataloader)
    return test_loss, test_acc

In [11]:
import copy
import csv
import os

# ──────────────────────────────────────────────────────────────────────────────
# CHECKPOINT HELPERS
# ──────────────────────────────────────────────────────────────────────────────

def save_checkpoint(epoch, model, optimizer, scheduler, results, best_acc, best_weights, path, curves_path=None, timestamp=None):
    """Zapisuje pełny stan trenowania do pliku .ckpt na Google Drive.
    Opcjonalnie aktualizuje curves.csv o epoki od ostatniego zapisu.
    """
    checkpoint = {
        "epoch":              epoch,
        "model_state":        model.state_dict(),
        "optimizer_state":    optimizer.state_dict(),
        "scheduler_state":    scheduler.state_dict() if scheduler is not None else None,
        "results":            results,
        "best_acc_test":      best_acc,
        "best_model_weights": best_weights,
    }
    torch.save(checkpoint, path)
    print(f"  [Checkpoint] Zapisano epoka {epoch+1} -> {path}")

    # Zapis curves.csv — dopisuje wiersze od last_saved_epoch do epoch
    if curves_path is not None and timestamp is not None:
        curves_header = ["timestamp", "epoch", "train_loss", "train_acc", "val_loss", "val_acc"]
        curves_exists = os.path.isfile(curves_path)
        # Sprawdź ile epok już zapisano dla tego timestamp
        already_saved = 0
        if curves_exists:
            with open(curves_path, "r") as f:
                already_saved = sum(1 for row in csv.DictReader(f) if str(row["timestamp"]) == str(timestamp))
        with open(curves_path, "a", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=curves_header)
            if not curves_exists:
                writer.writeheader()
            for ep in range(already_saved, epoch + 1):
                writer.writerow({
                    "timestamp":  timestamp,
                    "epoch":      ep + 1,
                    "train_loss": round(results["train_loss"][ep], 6),
                    "train_acc":  round(results["train_acc"][ep],  6),
                    "val_loss":   round(results["test_loss"][ep],  6),
                    "val_acc":    round(results["test_acc"][ep],   6),
                })
        print(f"  [Checkpoint] Zaktualizowano curves.csv (epoki {already_saved+1}-{epoch+1})")


def load_checkpoint(path, model, optimizer, scheduler):
    """
    Wczytuje checkpoint.
    Zwraca (start_epoch, results, best_acc, best_weights).
    Modyfikuje model/optimizer/scheduler IN-PLACE.
    """
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    if scheduler is not None and ckpt["scheduler_state"] is not None:
        scheduler.load_state_dict(ckpt["scheduler_state"])
    print(f"  [Checkpoint] Wznowienie od epoki {ckpt['epoch']+2} (ostatnia zapisana: {ckpt['epoch']+1})")
    return ckpt["epoch"] + 1, ckpt["results"], ckpt["best_acc_test"], ckpt["best_model_weights"]


# ──────────────────────────────────────────────────────────────────────────────
# GŁÓWNA FUNKCJA TRENINGOWA (z checkpointingiem i wznowieniem)
# ──────────────────────────────────────────────────────────────────────────────

def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader: torch.utils.data.DataLoader,
          optimizer: torch.optim.Optimizer,
          loss_fn: torch.nn.Module,
          epochs: int,
          save_dir: str,
          scheduler=None,
          model_name: str = "model",
          timestamp: int = None,
          checkpoint_every: int = 5,
          resume_from: str = None,
          curves_path: str = None):
    """
    Trenuje model z automatycznym checkpointingiem.

    Parametry:
        checkpoint_every : co ile epok zapisywać checkpoint (domyślnie 5)
        resume_from      : ścieżka do pliku .ckpt z poprzedniej sesji;
                           jeśli None - trening zaczyna od zera
    """

    results = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}
    best_acc_test = 0
    best_model_weights = copy.deepcopy(model.state_dict())
    start_epoch = 0

    # ── Wznowienie z checkpointu ──────────────────────────────────────────────
    if resume_from is not None and os.path.isfile(resume_from):
        start_epoch, results, best_acc_test, best_model_weights = load_checkpoint(
            resume_from, model, optimizer, scheduler
        )
    elif resume_from is not None:
        print(f"  [Checkpoint] Nie znaleziono pliku {resume_from}, trening od zera.")

    # Ścieżka do checkpointu (nadpisywana co 5 epok)
    checkpoint_path = os.path.join(save_dir, "checkpoint_latest.ckpt")

    # ── Pętla treningowa ──────────────────────────────────────────────────────
    for epoch in tqdm(range(start_epoch, epochs)):
        train_loss, train_acc = train_step(
            model=model, dataloader=train_dataloader,
            loss_fn=loss_fn, optimizer=optimizer
        )
        test_loss, test_acc = test_step(
            model=model, dataloader=test_dataloader, loss_fn=loss_fn
        )

        if scheduler is not None:
            scheduler.step()

        if test_acc > best_acc_test:
            best_acc_test = test_acc
            best_model_weights = copy.deepcopy(model.state_dict())

        print(
            f"Epoch: {epoch+1} | "
            f"train_loss: {train_loss:.4f} | "
            f"train_acc: {train_acc:.4f} | "
            f"valid_loss: {test_loss:.4f} | "
            f"valid_acc: {test_acc:.4f}"
        )

        results["train_loss"].append(train_loss.item() if isinstance(train_loss, torch.Tensor) else train_loss)
        results["train_acc"].append(train_acc.item()  if isinstance(train_acc,  torch.Tensor) else train_acc)
        results["test_loss"].append(test_loss.item()  if isinstance(test_loss,  torch.Tensor) else test_loss)
        results["test_acc"].append(test_acc.item()   if isinstance(test_acc,   torch.Tensor) else test_acc)

        # ── Zapis checkpointu co checkpoint_every epok ────────────────────────
        if (epoch + 1) % checkpoint_every == 0:
            save_checkpoint(
                epoch, model, optimizer, scheduler,
                results, best_acc_test, best_model_weights,
                checkpoint_path,
                curves_path=curves_path,
                timestamp=timestamp,
            )
    final_model_path = os.path.join(save_dir, f"{model_name}_{timestamp}.pt")
    torch.save(best_model_weights, final_model_path)
    print(f"  [Trening zakończony] Najlepszy model -> {final_model_path}")

    # Usuń checkpoint po zakończonym treningu (opcjonalne - zakomentuj jeśli chcesz zachować)
    if os.path.isfile(checkpoint_path):
        os.remove(checkpoint_path)
        print(f"  [Checkpoint] Usunięto tymczasowy checkpoint: {checkpoint_path}")

    results["best_val_acc"] = best_acc_test
    return results


def save_results_to_csv(results, config, curves_path, experiments_path):
    """Zapisuje wyniki do dwoch plikow CSV.

    curves_path      — krzywe uczenia (jedna linia per epoka)
    experiments_path — podsumowanie eksperymentu (jedna linia per run)
    """
    timestamp = config["timestamp"]
    num_epochs = len(results["train_loss"])

    # ── 1. curves.csv ─────────────────────────────────────────────────────────
    curves_header = ["timestamp", "epoch", "train_loss", "train_acc", "val_loss", "val_acc"]
    curves_exists = os.path.isfile(curves_path)
    # Sprawdź ile epok tego runu już zapisano (przez checkpointy)
    already_saved = 0
    if curves_exists:
        with open(curves_path, "r") as f:
            already_saved = sum(1 for row in csv.DictReader(f) if str(row["timestamp"]) == str(timestamp))
    with open(curves_path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=curves_header)
        if not curves_exists:
            writer.writeheader()
        for epoch in range(already_saved, num_epochs):
            writer.writerow({
                "timestamp":  timestamp,
                "epoch":      epoch + 1,
                "train_loss": round(results["train_loss"][epoch], 6),
                "train_acc":  round(results["train_acc"][epoch],  6),
                "val_loss":   round(results["test_loss"][epoch],  6),
                "val_acc":    round(results["test_acc"][epoch],   6),
            })

    # ── 2. experiments.csv ────────────────────────────────────────────────────
    exp_header = [
        "timestamp", "model", "pretrained_weights", "optimizer",
        "learning_rate", "batch_size", "num_epochs", "weight_decay",
        "dropout", "scheduler", "data_augmentation",
        "best_val_acc", "training_time_s"
    ]
    exp_exists = os.path.isfile(experiments_path)
    with open(experiments_path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=exp_header)
        if not exp_exists:
            writer.writeheader()
        writer.writerow({
            "timestamp":          timestamp,
            "model":              config["model"],
            "pretrained_weights": config["pretrained_weights"],
            "optimizer":          config["optimizer"],
            "learning_rate":      config["learning_rate"],
            "batch_size":         config["batch_size"],
            "num_epochs":         num_epochs,
            "weight_decay":       config["weight_decay"],
            "dropout":            config["dropout"],
            "scheduler":          config["scheduler"],
            "data_augmentation":  config["data_augmentation"],
            "best_val_acc":       round(results["best_val_acc"], 6),
            "training_time_s":    round(config["training_time"], 2),
        })

    print(f"Zapisano krzywe  -> {curves_path}")
    print(f"Zapisano exp     -> {experiments_path}")


In [12]:
# take 75% of train/valid data
BATCH_SIZE = 64  # fix: 32 za malo dla SGD z momentum, standard to 128
NUM_WORKERS = 2   # fix #10: os.cpu_count() niestabilne na Colab

class_indices_train = defaultdict(list)
class_indices_valid = defaultdict(list)

for i, label in enumerate(train_data.targets):
    class_indices_train[label].append(i)

for i, label in enumerate(valid_data.targets):
    class_indices_valid[label].append(i)

selected_train = []
selected_valid = []
for label, inds in class_indices_train.items():
    random.shuffle(inds)
    selected_train.extend(inds[:int(0.75 * len(inds))])

for label, inds in class_indices_valid.items():
    random.shuffle(inds)
    selected_valid.extend(inds[:int(0.75 * len(inds))])

subset_train = Subset(train_data, selected_train)
subset_valid = Subset(valid_data, selected_valid)

train_dataloader = DataLoader(
    subset_train,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    shuffle=True
)

valid_dataloader = DataLoader(
    subset_valid,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    shuffle=False  # fix #5: walidacja nie powinna byc shufflowana
)


In [ ]:
# uzupelnic przed puszczeniem
MODEL = "densenet121"
BATCH_SIZE = 64
NUM_WORKERS = 2
NUM_EPOCHS = 40
LEARNING_RATE = 0.01
WEIGHT_DECAY = 1e-4
DROPOUT = 0
PRETRAINED_WEIGHTS = "IMAGENET1K_V1"
DATA_AUG = "RandomCrop+HorizontalFlip+Normalize"
SCHEDULER_NAME = "None"

# ── Ścieżki zapisu ────────────────────────────────────────────────────────────
SAVE_DIR = "/content/drive/MyDrive/data"

if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)
    print(f"Utworzono brakujący folder: {SAVE_DIR}")

CURVES_CSV      = f"{SAVE_DIR}/curves.csv"
EXPERIMENTS_CSV = f"{SAVE_DIR}/experiments.csv"

# ── Wznowienie trenowania ─────────────────────────────────────────────────────
# Ustaw RESUME_FROM na ścieżkę do pliku .ckpt jeśli chcesz wznowić trening.
# Jeśli trenujesz od zera, zostaw None.
#
#   RESUME_FROM = None                                    # trening od zera
#   RESUME_FROM = f"{SAVE_DIR}/checkpoint_latest.ckpt"   # wznowienie
#
RESUME_FROM = None  # <-- zmień na ścieżkę .ckpt, żeby wznowić trening

# ── Budowa modelu ─────────────────────────────────────────────────────────────
model = build_densenet121(10, weights=PRETRAINED_WEIGHTS, dropout=DROPOUT).to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.SGD(
    model.parameters(),
    momentum=0.9,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Scheduler opcjonalny - odkomentuj jesli chcesz uzyc
# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
# SCHEDULER_NAME = "CosineAnnealingLR"
scheduler = None

timestamp = int(datetime.now().timestamp())

# ── Uruchomienie treningu ─────────────────────────────────────────────────────
start_time = timer()
model_results = train(
    model=model,
    train_dataloader=train_dataloader,
    test_dataloader=valid_dataloader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    epochs=NUM_EPOCHS,
    save_dir=SAVE_DIR,
    scheduler=scheduler,
    checkpoint_every=5,       # zapis co 5 epok
    model_name=MODEL,         # nazwa modelu w pliku .pt
    timestamp=timestamp,      # timestamp w pliku .pt
    resume_from=RESUME_FROM,  # None = od zera; ścieżka .ckpt = wznowienie
    curves_path=CURVES_CSV,   # aktualizacja curves.csv co checkpoint
)
end_time = timer()
print(f"Total training time: {end_time - start_time:.3f} seconds")

# ── Zapis wyników do CSV ──────────────────────────────────────────────────────
config = {
    "timestamp":          timestamp,
    "model":              MODEL,
    "pretrained_weights": PRETRAINED_WEIGHTS,
    "optimizer":          "SGD",
    "learning_rate":      LEARNING_RATE,
    "batch_size":         BATCH_SIZE,
    "weight_decay":       WEIGHT_DECAY,
    "dropout":            DROPOUT,
    "scheduler":          SCHEDULER_NAME,
    "data_augmentation":  DATA_AUG,
    "training_time":      end_time - start_time,
}

save_results_to_csv(
    results=model_results,
    config=config,
    curves_path=CURVES_CSV,
    experiments_path=EXPERIMENTS_CSV
)
runtime.unassign()


  0%|          | 0/40 [00:00<?, ?it/s]